# NB6 — Assemble every table and figure for the paper

Reads the artifacts written by NB1–NB5 and emits the LaTeX the paper `\input`s,
so no number is ever retyped by hand. Every table carries the `n` it was computed
over and the operating point it was measured at.

Run this last. It needs no GPU and finishes in under a minute.

**Emits** into `ART/paper/`: `tab_*.tex`, `fig_*.pdf`, and `claims.json` — a
machine-checkable list of every numeric claim the paper makes, with the artifact
each one came from. Section 4 verifies that the paper's `.tex` contains no number
that `claims.json` cannot account for.

In [ ]:
import os, sys, json, math, time, random, hashlib, re, gc, warnings, shutil, glob, zipfile, io
from pathlib import Path
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# torch is required by the experiment notebooks but not by the reporting one,
# so a missing install degrades to a clear message rather than a traceback.
try:
    import torch
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    if DEVICE == "cuda":
        props = torch.cuda.get_device_properties(0)
        print(f"GPU: {props.name} | {props.total_memory/1e9:.1f} GB | "
              f"n_gpus={torch.cuda.device_count()}")
    else:
        print("WARNING: no GPU detected. Everything will run but ~10-20x slower.")
except ImportError:
    torch = None
    DEVICE = "cpu"
    print("torch not installed (fine for the reporting notebook, required for the rest).")

# Artifact directory. On Kaggle write to /kaggle/working so results persist in
# the version output; on Colab mount Drive if you want results to survive a
# disconnect.
if Path("/kaggle/working").exists():
    ART = Path("/kaggle/working/cognisync_tmlr")
elif Path("/content/drive/MyDrive/cognisync_tmlr").exists():
    ART = Path("/content/drive/MyDrive/cognisync_tmlr")
elif Path("/content/drive/MyDrive").exists():
    ART = Path("/content/drive/MyDrive/cognisync_tmlr")
else:
    ART = Path("./cognisync_tmlr").resolve()

(ART / "results").mkdir(parents=True, exist_ok=True)
(ART / "cache").mkdir(parents=True, exist_ok=True)


def discover_artifacts():
    """Universal artifact finder: searches /kaggle/input, /kaggle/working, /content,
    parent folders, and all subdirectories for uploaded .zip archives or raw result files,
    extracting/copying them directly into ART / 'results'."""
    res_dir = ART / "results"
    res_dir.mkdir(parents=True, exist_ok=True)
    
    search_dirs = [
        Path("/kaggle/input"), Path("/kaggle/working"),
        Path("/content"), Path("/content/drive/MyDrive"),
        Path("."), Path(".."), Path("../.."),
        Path("tmlr"), Path("../tmlr"), Path("../../tmlr"),
        Path("cognisync_tmlr"), Path("../cognisync_tmlr"), Path("../../cognisync_tmlr"),
        Path("results"), Path("../results"), Path("../../results"),
        Path("tmlr/results"), Path("../tmlr/results"), Path("../../tmlr/results"),
    ]

    seen = set()
    for sdir in search_dirs:
        try:
            if not sdir.exists():
                continue
            resolved = sdir.resolve()
            if resolved in seen:
                continue
            seen.add(resolved)
        except Exception:
            continue

        # 1. Search for any .zip files
        for zpath in sdir.glob("*.zip"):
            try:
                with zipfile.ZipFile(zpath, "r") as zf:
                    extracted = 0
                    for info in zf.infolist():
                        if info.is_dir():
                            continue
                        fname = Path(info.filename).name
                        if (fname.startswith(("nb", "fig_", "beir")) or fname.endswith((".parquet", ".csv", ".json", ".tex", ".pdf", ".png"))):
                            target_path = res_dir / fname
                            if not target_path.exists():
                                with zf.open(info) as source, open(target_path, "wb") as target:
                                    shutil.copyfileobj(source, target)
                                extracted += 1
                    if extracted > 0:
                        print(f">>> Unpacked {extracted} artifacts from {zpath.name} -> {res_dir}")
            except Exception:
                pass

        # 2. Search for any raw files in input datasets / working dirs
        for p in sdir.rglob("nb*.*"):
            if p.is_file() and p.suffix in [".parquet", ".csv", ".json", ".tex", ".pdf", ".png"]:
                dest = res_dir / p.name
                if not dest.exists():
                    try:
                        shutil.copy(p, dest)
                        print(f">>> Found & copied artifact: {p.name} -> {dest}")
                    except Exception:
                        pass

        for p in sdir.rglob("fig_*.*"):
            if p.is_file() and p.suffix in [".pdf", ".png"]:
                dest = res_dir / p.name
                if not dest.exists():
                    try:
                        shutil.copy(p, dest)
                        print(f">>> Found & copied figure: {p.name} -> {dest}")
                    except Exception:
                        pass


discover_artifacts()
print("Artifacts directory ->", ART)


def save_json(obj, name):
    p = ART / "results" / name
    with open(p, "w") as f:
        json.dump(obj, f, indent=2, default=float)
    print("saved", p)
    return p


def save_csv(df, name):
    p = ART / "results" / name
    df.to_csv(p, index=False)
    print("saved", p, df.shape)
    return p

# Robust resolution for PAPER directory
PAPER = ART / "paper"
for cand in [Path("tmlr/paper"), Path("../paper"), Path("../../tmlr/paper")]:
    if cand.exists():
        PAPER = cand.resolve()
        break
PAPER.mkdir(parents=True, exist_ok=True)

R = ART / "results"

# Discover and unpack artifacts from /kaggle/input, /kaggle/working, or /content
discover_artifacts()

def have(name):
    p = R / name
    return p if p.exists() else None

print("\n" + "="*60)
print("NB6 INPUT ARTIFACT AUDIT:")
print("="*60)
for n in ["nb1_summary.csv", "nb1_pairwise.csv", "nb2_headroom.csv",
          "nb2_policy_comparison.csv", "nb2_alpha_predictability.csv",
          "nb3_attack_defense_matrix.csv", "nb3_roc_auc.csv", "nb3_calibration.csv",
          "nb4_behavioural_summary.csv", "nb4_retrieval_vs_behavioural.csv",
          "nb4_tool_selection_summary.csv", "nb5_cost_quality.csv"]:
    print(f"  [{'✓' if have(n) else '✗'}] {n}")
print("="*60 + "\n")

CLAIMS = {}
def claim(key, value, source, note=""):
    CLAIMS[key] = {"value": value, "source": source, "note": note}
    return value

In [ ]:
def tex_escape(s):
    return str(s).replace("_", r"\_").replace("&", r"\&").replace("%", r"\%")


def validate_tabular(body, name):
    """Fail loudly on the two LaTeX errors that are easy to generate and painful
    to debug: a row whose cell count disagrees with the column spec, and a stray
    unescaped percent sign (which silently comments out the rest of a line)."""
    key = "begin{tabular}{"
    i = body.find(key)
    assert i >= 0, f"{name}: no tabular spec found"
    j = body.index("}", i + len(key))
    ncol = sum(1 for ch in body[i + len(key):j] if ch in "lcr")
    problems = []
    for i, line in enumerate(body.splitlines()):
        st = line.strip()
        if not st.endswith("\\\\") or st.startswith("%"):
            continue
        if "multicolumn" in st:
            continue
        cells = st[:-2].split("&")
        if len(cells) != ncol:
            problems.append(f"    line {i}: {len(cells)} cells, spec says {ncol}: {st[:78]}")
    for i, line in enumerate(body.splitlines()):
        for k, ch in enumerate(line):
            if ch == "%" and (k == 0 or line[k - 1] != chr(92)):
                problems.append(f"    line {i}: unescaped % -> {line[:78]}")
    assert not problems, f"{name} is malformed:\n" + "\n".join(problems)
    return ncol


def write_tex(name, body, caption, label):
    ncol = validate_tabular(body, name)
    for m in re.finditer(r"%", caption):
        assert m.start() > 0 and caption[m.start() - 1] == chr(92), \
            f"{name}: unescaped % in caption -> {caption[:90]}"
    doc = ("\\begin{table}[t]\n\\centering\n\\small\n"
           f"\\caption{{{caption}}}\n\\label{{{label}}}\n{body}\n\\end{{table}}\n")
    (PAPER / name).write_text(doc)
    print(f"wrote {PAPER / name}  ({ncol} columns, validated)")


# ---------------------------------------------------------------- Table 1
if have("nb1_summary.csv"):
    s = pd.read_csv(R / "nb1_summary.csv")
    enc = s.encoder.iloc[0]
    s = s[s.encoder == enc]
    order = ["bm25", "dense", "rrf", "alpha_fixed", "alpha_learned",
             "alpha_learned_override", "alpha_oracle"]
    pretty = {"bm25": "BM25", "dense": "Dense", "rrf": "RRF ($k{=}60$)",
              "alpha_fixed": "Fixed $\\alpha^\\star$", "alpha_learned": "Learned $\\alpha$",
              "alpha_learned_override": "Learned $\\alpha$ + dense override",
              "alpha_oracle": "\\textit{Oracle} $\\alpha$"}
    ds_list = sorted(s.dataset.unique())
    budgets = sorted(s.budget.unique())
    L = ["\\begin{tabular}{l" + "c" * (len(ds_list) + 1) + "}", "\\toprule",
         "First stage & " + " & ".join(tex_escape(d) for d in ds_list) + " & Mean \\\\"]
    for b in budgets:
        L += ["\\midrule",
              "\\multicolumn{%d}{l}{\\textit{%s}} \\\\" %
              (len(ds_list) + 2, "no reranking" if b == 0 else f"cross-encoder top-{b}")]
        for sysname in order:
            r = s[(s.system == sysname) & (s.budget == b)]
            if r.empty:
                continue
            cells = []
            for d in ds_list:
                v = r[r.dataset == d]["ndcg10"]
                cells.append(f"{float(v.iloc[0]):.3f}" if len(v) else "--")
            L.append(f"{pretty[sysname]} & " + " & ".join(cells) +
                     f" & {r.ndcg10.mean():.3f} \\\\")
            claim(f"ndcg10.{sysname}.budget{b}", round(float(r.ndcg10.mean()), 4),
                  "nb1_summary.csv", f"mean over {len(ds_list)} BEIR corpora, encoder={enc}")
    L += ["\\bottomrule", "\\end{tabular}"]
    write_tex("tab_main_retrieval.tex", "\n".join(L),
              "Full-corpus nDCG@10 on BEIR under a matched cross-encoder budget. "
              "Every first stage is scored at the same reranking depth, so the "
              "fusion mechanism is compared against dense retrieval on equal "
              "compute. Oracle $\\alpha$ is an upper bound, not a system.",
              "tab:main_retrieval")

# ---------------------------------------------------------------- Table 2
if have("nb2_headroom.csv") and have("nb2_policy_comparison.csv"):
    h = pd.read_csv(R / "nb2_headroom.csv")
    p = pd.read_csv(R / "nb2_policy_comparison.csv")
    L = ["\\begin{tabular}{lcccc}", "\\toprule",
         "Policy & nDCG@10 & $\\Delta$ vs.\\ Dense & 95\\% CI & \\% headroom \\\\",
         "\\midrule"]
    fixed = p[p.policy.str.startswith("Fixed")]["ndcg10"]
    orac = p[p.policy.str.startswith("Oracle")]["ndcg10"]
    span = (float(orac.iloc[0]) - float(fixed.iloc[0])) if len(fixed) and len(orac) else np.nan
    for _, r in p.iterrows():
        pct = ((r.ndcg10 - float(fixed.iloc[0])) / span * 100) if span and span > 1e-9 else np.nan
        L.append(f"{tex_escape(r.policy)} & {r.ndcg10:.4f} & {r.delta_vs_dense:+.4f} & "
                 f"[{r.ci_low:+.4f}, {r.ci_high:+.4f}] & "
                 + (f"{pct:.0f}\\%" if np.isfinite(pct) else "--") + " \\\\")
    L += ["\\bottomrule", "\\end{tabular}"]
    write_tex("tab_alpha_policies.tex", "\n".join(L),
              "Per-query fusion policies against the oracle ceiling. "
              "``\\% headroom'' is the share of the oracle-minus-fixed gap a policy "
              "captures. Intervals are paired bootstrap over per-query nDCG@10.",
              "tab:alpha_policies")
    claim("headroom.oracle_minus_fixed", round(float(span), 4) if span == span else None,
          "nb2_policy_comparison.csv")
    claim("headroom.pct_queries_alpha_irrelevant",
          round(float(h.pct_queries_alpha_irrelevant.mean()), 4), "nb2_headroom.csv",
          "fraction of queries where the entire alpha grid moves nDCG@10 by <0.01")

if have("nb2_alpha_predictability.csv"):
    pr = pd.read_csv(R / "nb2_alpha_predictability.csv")
    L = ["\\begin{tabular}{lccc}", "\\toprule",
         "Model & CV $R^2$ & Spearman $\\rho$ & MAE \\\\", "\\midrule"]
    for _, r in pr.iterrows():
        L.append(f"{tex_escape(r.model)} & {r.cv_r2:.3f} & {r.spearman_rho:.3f} & {r.mae:.3f} \\\\")
    L += ["\\bottomrule", "\\end{tabular}"]
    write_tex("tab_alpha_predictability.tex", "\n".join(L),
              "How predictable the oracle fusion weight is from the six query and "
              "score-distribution features, under 5-fold cross-validation.",
              "tab:alpha_predictability")

In [ ]:
# ---------------------------------------------------------------- Table 3
if have("nb3_attack_defense_matrix.csv"):
    m = pd.read_csv(R / "nb3_attack_defense_matrix.csv")
    f_main = 0.01 if (m.target_fpr == 0.01).any() else float(m.target_fpr.min())
    mm = m[m.target_fpr == f_main]
    piv = mm.pivot(index="attack", columns="defense", values="asr")
    cols = [c for c in ["D0_none", "D1_3feat_tiny", "D1b_3feat_trained", "D2_embed_probe",
                        "D3_distilbert", "D4_guard_zeroshot", "D5_perplexity", "D6_ensemble"]
            if c in piv.columns]
    piv = piv[cols]
    short = {"D0_none": "none", "D1_3feat_tiny": "D1", "D1b_3feat_trained": "D1b",
             "D2_embed_probe": "D2", "D3_distilbert": "D3", "D4_guard_zeroshot": "D4",
             "D5_perplexity": "D5", "D6_ensemble": "D6"}
    L = ["\\begin{tabular}{l" + "c" * len(cols) + "}", "\\toprule",
         "Attacker & " + " & ".join(short[c] for c in cols) + " \\\\", "\\midrule"]
    for a in piv.index:
        L.append(tex_escape(a) + " & " +
                 " & ".join(f"{piv.loc[a, c]:.2f}" for c in cols) + " \\\\")
        for c in cols:
            claim(f"asr.{a}.{c}.fpr{f_main}", round(float(piv.loc[a, c]), 4),
                  "nb3_attack_defense_matrix.csv", f"n={int(mm[mm.attack==a].n.iloc[0])}")
    L += ["\\bottomrule", "\\end{tabular}"]
    # The "none" column already is the undefended entry rate, so it is named in
    # the caption rather than repeated as a row.
    write_tex("tab_attack_defense.tex", "\n".join(L),
              f"Retrieval-level attack success rate at a matched false-positive "
              f"budget of {f_main*100:g}\\%. Rows are attackers ordered by capability; "
              f"columns are defenses ordered by cost. The \\textit{{none}} column is "
              f"the undefended payload-entry rate. Every defended cell is measured "
              f"at the same operating point, calibrated on held-out clean documents.",
              "tab:attack_defense")

    ucost = (m[m.attack == "A0_static_templates"]
             .pivot(index="defense", columns="target_fpr", values="utility_cost_ndcg"))
    L = ["\\begin{tabular}{l" + "c" * len(ucost.columns) + "}", "\\toprule",
         "Defense & " + " & ".join(f"FPR {c*100:g}\\%" for c in ucost.columns) +
         " \\\\", "\\midrule"]
    for d in ucost.index:
        L.append(tex_escape(d) + " & " +
                 " & ".join(f"{ucost.loc[d, c]:.4f}" for c in ucost.columns) + " \\\\")
    L += ["\\bottomrule", "\\end{tabular}"]
    write_tex("tab_utility_cost.tex", "\n".join(L),
              "Utility cost of filtering: nDCG@10 lost on clean full-corpus "
              "retrieval, with no attack present, at each operating point.",
              "tab:utility_cost")

# ---------------------------------------------------------------- Table 4
if have("nb4_retrieval_vs_behavioural.csv"):
    g = pd.read_csv(R / "nb4_retrieval_vs_behavioural.csv")
    L = ["\\begin{tabular}{llccc}", "\\toprule",
         "Model & Attacker & ASR$_\\text{retr}$ & ASR$_\\text{behav}$ & "
         "$P(\\text{comply}\\mid\\text{entry})$ \\\\", "\\midrule"]
    for _, r in g.iterrows():
        L.append(f"{tex_escape(r.model.split('/')[-1])} & {tex_escape(r.attack)} & "
                 f"{r.asr_retrieval_undefended:.2f} & {r.asr_behavioural_undefended:.2f} & "
                 f"{r.compliance_given_entry:.2f} \\\\")
        claim(f"compliance.{r.model.split('/')[-1]}.{r.attack}",
              round(float(r.compliance_given_entry), 4),
              "nb4_retrieval_vs_behavioural.csv")
    L += ["\\bottomrule", "\\end{tabular}"]
    write_tex("tab_behavioural_gap.tex", "\n".join(L),
              "Retrieval-level payload entry versus downstream compliance, on the "
              "same episodes with no defense. The last column is the factor by "
              "which a retrieval-level metric must be discounted to estimate "
              "real exposure.",
              "tab:behavioural_gap")

# ---------------------------------------------------------------- Table 5
if have("nb5_cost_quality.csv"):
    c = pd.read_csv(R / "nb5_cost_quality.csv")
    c = c[c.depth == c.depth.max()]
    L = ["\\begin{tabular}{lrrrrrr}", "\\toprule",
         "Corpus & $|D|$ & CE/query & nDCG@10 & p50 & p95 & p99 \\\\", "\\midrule"]
    for _, r in c.iterrows():
        L.append(f"{tex_escape(r.dataset)} & {int(r.n_docs):,} & "
                 f"{int(r.ce_forward_passes_per_query)} & {r.ndcg10:.3f} & "
                 f"{r.p50_ms:.0f} & {r.p95_ms:.0f} & {r.p99_ms:.0f} \\\\")
    L += ["\\bottomrule", "\\end{tabular}"]
    write_tex("tab_cost.tex", "\n".join(L),
              "Cost and quality from the same runs, on real corpora with a "
              "persistent index. Cross-encoder forward passes per query is the "
              "hardware-independent cost; latencies are on a single T4.",
              "tab:cost")

## Result macros for the manuscript

The paper's prose contains no hand-typed numbers. Every inline figure is a LaTeX
macro defined in `results_macros.tex`, generated here from the artifacts. Until
this notebook has run, each macro renders as a red placeholder in the compiled
PDF, so an unfilled slot is impossible to miss and impossible to mistake for a
measurement.

In [ ]:
def pct(x, d=1):
    return f"{100*float(x):.{d}f}\\%"


def num(x, d=3):
    return f"{float(x):.{d}f}"


M = {}

# ---- retrieval (NB1) -------------------------------------------------------
if have("nb1_summary.csv"):
    s1 = pd.read_csv(R / "nb1_summary.csv")
    s1 = s1[s1.encoder == s1.encoder.iloc[0]]
    top_budget = int(s1.budget.max())
    def mean_ndcg(system, budget):
        v = s1[(s1.system == system) & (s1.budget == budget)]["ndcg10"]
        return float(v.mean()) if len(v) else float("nan")
    M["ndcgDenseCE"] = num(mean_ndcg("dense", top_budget))
    M["ndcgLearnedCE"] = num(mean_ndcg("alpha_learned", top_budget))
    claim("macro.ndcgDenseCE", mean_ndcg("dense", top_budget), "nb1_summary.csv",
          f"mean nDCG@10, cross-encoder top-{top_budget}")
    claim("macro.ndcgLearnedCE", mean_ndcg("alpha_learned", top_budget),
          "nb1_summary.csv", f"mean nDCG@10, cross-encoder top-{top_budget}")

if have("nb1_corpus_stats.csv"):
    cs = pd.read_csv(R / "nb1_corpus_stats.csv")
    M["nBeirCorpora"] = str(len(cs))
    M["nBeirDocs"] = f"{int(cs.n_docs.sum()):,}"
    M["nBeirQueries"] = f"{int(cs.n_queries.sum()):,}"

# ---- fusion headroom (NB2) -------------------------------------------------
if have("nb2_policy_comparison.csv"):
    p2 = pd.read_csv(R / "nb2_policy_comparison.csv")
    orac = p2[p2.policy.str.startswith("Oracle")]["ndcg10"]
    fixed = p2[p2.policy.str.startswith("Fixed")]["ndcg10"]
    learned = p2[p2.policy.str.startswith("Learned")]["ndcg10"]
    if len(orac) and len(fixed):
        M["ndcgOracle"] = num(orac.iloc[0]); M["ndcgFixed"] = num(fixed.iloc[0])
        span = float(orac.iloc[0]) - float(fixed.iloc[0])
        if len(learned) and abs(span) > 1e-9:
            frac = (float(learned.max()) - float(fixed.iloc[0])) / span
            M["headroomPct"] = pct(max(0.0, frac), 0)
            claim("macro.headroomPct", round(frac, 4), "nb2_policy_comparison.csv",
                  "share of oracle-minus-fixed gap captured by the best learned policy")

if have("nb2_headroom.csv"):
    h2 = pd.read_csv(R / "nb2_headroom.csv")
    M["flatFrac"] = pct(h2.pct_queries_alpha_irrelevant.mean(), 0)

if have("nb2_alpha_predictability.csv"):
    a2 = pd.read_csv(R / "nb2_alpha_predictability.csv")
    M["alphaRtwo"] = num(a2.cv_r2.max(), 3)
    claim("macro.alphaRtwo", float(a2.cv_r2.max()), "nb2_alpha_predictability.csv",
          "best cross-validated R^2 over all predictors tried")

# ---- security (NB3) --------------------------------------------------------
if have("nb3_attack_defense_matrix.csv"):
    m3 = pd.read_csv(R / "nb3_attack_defense_matrix.csv")
    f_main = 0.01 if (m3.target_fpr == 0.01).any() else float(m3.target_fpr.min())
    mm = m3[m3.target_fpr == f_main]
    def asr(attack, defense):
        v = mm[(mm.attack == attack) & (mm.defense == defense)]["asr"]
        return float(v.iloc[0]) if len(v) else float("nan")
    for key, (a, d) in {"asrDoneStatic": ("A0_static_templates", "D1_3feat_tiny"),
                        "asrDoneAdaptive": ("A5_score_guided", "D1_3feat_tiny"),
                        "asrDthreeAdaptive": ("A5_score_guided", "D3_distilbert")}.items():
        v = asr(a, d)
        if v == v:
            M[key] = pct(v, 1)
            claim(f"macro.{key}", round(v, 4), "nb3_attack_defense_matrix.csv",
                  f"{a} vs {d} at FPR {f_main:.1%}")
    uc = mm[(mm.attack == "A0_static_templates") &
            (mm.defense == "D3_distilbert")]["utility_cost_ndcg"]
    if len(uc):
        M["utilityCost"] = num(uc.iloc[0], 4)

# ---- behaviour (NB4) -------------------------------------------------------
if have("nb4_retrieval_vs_behavioural.csv"):
    g4 = pd.read_csv(R / "nb4_retrieval_vs_behavioural.csv")
    g4 = g4[np.isfinite(g4.compliance_given_entry)]
    if len(g4):
        M["complianceRate"] = pct(g4.compliance_given_entry.mean(), 0)
        fin = g4[np.isfinite(g4.overstatement_factor)]
        if len(fin):
            M["overstatement"] = f"{fin.overstatement_factor.mean():.1f}"
        claim("macro.complianceRate", round(float(g4.compliance_given_entry.mean()), 4),
              "nb4_retrieval_vs_behavioural.csv", "mean over models and attacks")

if have("nb4_behavioural_summary.csv"):
    b4 = pd.read_csv(R / "nb4_behavioural_summary.csv")
    hd = b4[(b4.system == "hardened") & (b4.attack != "clean") & (b4.position == 0)]
    if len(hd):
        M["promptHardenASR"] = pct(hd.asr_behavioural.mean(), 1)
        claim("macro.promptHardenASR", round(float(hd.asr_behavioural.mean()), 4),
              "nb4_behavioural_summary.csv", "hardened system prompt, poison at rank 0")

lines = ["%% Auto-generated by NB6. Do not edit by hand.",
         "%% Regenerate by re-running NB6 after the experiment notebooks."]
for k, v in sorted(M.items()):
    lines.append("\\newcommand{\\%s}{%s}" % (k, v))
(PAPER / "results_macros.tex").write_text("\n".join(lines) + "\n")
print(f"wrote {PAPER/'results_macros.tex'} with {len(M)} macros\n")
print("\n".join(lines[2:]))

missing = [k for k in ["ndcgDenseCE", "ndcgLearnedCE", "ndcgOracle", "ndcgFixed",
                       "headroomPct", "flatFrac", "alphaRtwo", "asrDoneStatic",
                       "asrDoneAdaptive", "asrDthreeAdaptive", "complianceRate",
                       "overstatement", "utilityCost", "promptHardenASR",
                       "nBeirCorpora", "nBeirDocs", "nBeirQueries"] if k not in M]
if missing:
    print("\nSTILL PLACEHOLDERS (run the notebook that produces each):")
    for k in missing:
        print("   \\" + k)
else:
    print("\nEvery macro the manuscript uses is filled.")

In [ ]:
import shutil
for f in R.glob("fig_*.pdf"):
    shutil.copy(f, PAPER / f.name)
for f in R.glob("fig_*.png"):
    shutil.copy(f, PAPER / f.name)

with open(PAPER / "claims.json", "w") as fh:
    json.dump(CLAIMS, fh, indent=2, default=float)
print(f"\n{len(CLAIMS)} numeric claims recorded in {PAPER/'claims.json'}")
for k, v in sorted(CLAIMS.items())[:25]:
    print(f"  {k:52s} = {v['value']}   <- {v['source']}")
print("\nFiles for the paper:")
for f in sorted(PAPER.iterdir()):
    print("  ", f.name)

## Claim audit

`claims.json` is the artifact that makes the paper checkable. Every number in the
manuscript should be traceable to a key in it, and the check below flags any
number in the `.tex` that is not — the direct answer to the reviewer complaint
that the CIKM version's tables disagreed with each other (Table 7 reporting FPR
1.04% while Table 9 reported 0.0% for the same system, and the repository's own
`latency_results.txt` recording a 3,325 ms median against the paper's 275.6 ms).

Point `PAPER_TEX` at the manuscript and run it before every submission.

In [ ]:
PAPER_TEX = None
for cand in [Path("cognisync_tmlr.tex"), Path("tmlr/paper/cognisync_tmlr.tex"),
             Path("../paper/cognisync_tmlr.tex"), Path("../../tmlr/paper/cognisync_tmlr.tex"),
             PAPER / "cognisync_tmlr.tex"]:
    if cand.exists():
        PAPER_TEX = cand
        break

if PAPER_TEX and PAPER_TEX.exists():
    src = PAPER_TEX.read_text()
    src = re.sub(r"%.*", "", src)
    nums = set(re.findall(r"(?<![\w.])(\d+\.\d{2,4})(?![\w])", src))
    known = set()
    for v in CLAIMS.values():
        if isinstance(v["value"], (int, float)):
            for d in (2, 3, 4):
                known.add(f"{v['value']:.{d}f}")
                known.add(f"{v['value']*100:.{d}f}")
    unaccounted = sorted(n for n in nums if n not in known)
    print(f"{len(nums)} numeric literals in the manuscript; "
          f"{len(nums)-len(unaccounted)} match a recorded claim.")
    if unaccounted:
        print("\nNot traceable to claims.json (check each is a citation year, a "
              "hyperparameter, or a typo):")
        for n in unaccounted[:60]:
            print("   ", n)
else:
    print(f"{PAPER_TEX} not found - upload the manuscript next to this notebook "
          f"to run the audit.")

## 5. Archive and Download Outputs

Packages all results, LaTeX tables, figures, and macros into `cognisync_tmlr_results.zip` and initiates automatic download.

In [ ]:
import shutil
from IPython.display import FileLink, display, Javascript

out_dir = str(ART)
zip_name = "cognisync_tmlr_results"
zip_base = f"/kaggle/working/{zip_name}" if Path("/kaggle/working").exists() else f"./{zip_name}"

shutil.make_archive(zip_base, "zip", out_dir)
zip_file = f"{zip_base}.zip"
size_mb = os.path.getsize(zip_file) / (1024 * 1024)

print("\n" + "="*60)
print(f">>> ARCHIVE CREATED: {zip_file} ({size_mb:.2f} MB)")
print("="*60)

display(FileLink(os.path.basename(zip_file)))

try:
    from google.colab import files
    files.download(zip_file)
except Exception:
    try:
        js_code = f"""
            const a = document.createElement("a");
            a.href = "{os.path.basename(zip_file)}";
            a.download = "{os.path.basename(zip_file)}";
            document.body.appendChild(a);
            a.click();
            document.body.removeChild(a);
        """
        display(Javascript(js_code))
        print(">>> Automatic download triggered in browser.")
    except Exception:
        print(">>> Click the link above to download your results archive.")